In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Load data
train_df = pd.read_csv('train_nurse.csv')
test_df = pd.read_csv('test_nurse.csv', index_col='ID')

train_df.head()

,aline_flg,age,gender_num,weight_first,bmi,sapsi_first,sofa_first,service_unit,service_num,day_icu_intime,day_icu_intime_num,hour_icu_intime,day_28_flg,sepsis_flg,chf_flg,afib_flg,renal_flg,liver_flg,copd_flg,cad_flg,stroke_flg,mal_flg,resp_flg,map_1st,hr_1st,temp_1st,spo2_1st,abg_count,wbc_first,hgb_first,platelet_first,sodium_first,potassium_first,tco2_first,chloride_first,bun_first,creatinine_first,po2_first,pco2_first,iv_day_1
0,1,72.36841,1.0,75.0,29.912791,15,9,SICU,1,Friday,6,6,1.0,0,0,0,0,0,0,0,0,1,0,92.000000,86,95.900002,100,22,8.1,14.1,354.0,138.0,4.6,15.0,109.0,41.0,1.6,196.0,39.0,2230.875000
1,0,48.40044,0.0,NaN,NaN,14,5,MICU,0,Monday,2,4,1.0,0,0,0,0,0,0,0,0,0,1,85.333298,95,99.800003,100,2,14.0,8.3,367.0,133.0,3.7,19.0,98.0,25.0,0.9,473.0,32.0,2850.000000
2,1,48.97547,1.0,97.0,31.579563,13,11,SICU,1,Thursday,5,2,0.0,0,0,0,0,1,1,0,0,1,0,74.000000,63,98.059998,100,14,3.6,11.5,145.0,133.0,4.0,31.0,95.0,18.0,1.0,493.0,34.0,3682.880127
3,1,60.37829,0.0,51.0,25.207321,18,7,SICU,1,Saturday,7,22,0.0,0,0,0,0,0,0,0,0,0,0,31.000000,93,96.800003,100,9,20.4,9.4,200.0,144.0,3.2,20.0,120.0,13.0,0.7,473.0,34.0,213.119995
4,1,83.20917,1.0,76.4,26.060587,18,5,SICU,1,Tuesday,3,0,0.0,0,0,1,0,0,0,0,0,1,0,65.000000,75,96.000000,100,4,9.9,14.8,249.0,137.0,4.8,19.0,104.0,29.0,2.0,251.0,35.0,NaN


In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1335 entries, 0 to 1334
Data columns (total 40 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   aline_flg           1335 non-null   int64  
 1   age                 1335 non-null   float64
 2   gender_num          1334 non-null   float64
 3   weight_first        1255 non-null   float64
 4   bmi                 1001 non-null   float64
 5   sapsi_first         1335 non-null   int64  
 6   sofa_first          1335 non-null   int64  
 7   service_unit        1335 non-null   object 
 8   service_num         1335 non-null   int64  
 9   day_icu_intime      1335 non-null   object 
 10  day_icu_intime_num  1335 non-null   int64  
 11  hour_icu_intime     1335 non-null   int64  
 12  day_28_flg          100 non-null    float64
 13  sepsis_flg          1335 non-null   int64  
 14  chf_flg             1335 non-null   int64  
 15  afib_flg            1335 non-null   int64  
 16  renal_

# Subtasks 1-2

In [ ]:
#Subtask 1
answer_sub1 = dict(train_df.isna().sum()).get('day_28_flg')

#Subtask 2
def subtask2(df):
    df = df.copy()
    df = df.dropna(subset=['day_28_flg'])

    group1 = df[df['sofa_first'] > 10]
    group2 = df[df['sofa_first'] <= 10]

    mortality1 = (group1['day_28_flg'].mean()) * 100
    mortality2 = (group2['day_28_flg'].mean()) * 100
    return [mortality1, mortality2]

answer_sub2 = subtask2(train_df)

# Subtask 3

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

train_df = train_df.dropna(subset='day_28_flg')
X = train_df.drop('day_28_flg', axis=1)
y = train_df['day_28_flg']

#Preparations
X, test_df = X.drop('day_icu_intime_num', axis=1), test_df.drop('day_icu_intime_num', axis=1)

num_cols = X.select_dtypes(include=[np.number]).columns
cat_cols = X.select_dtypes(include=['object']).columns

def make_preprocessor():
    num_transformer = Pipeline(steps=[
        ('imputer', KNNImputer()),
        ('scaler', StandardScaler())
    ])
    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(sparse_output=False))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols),
    ],  verbose_feature_names_out=False)

    return preprocessor

preprocessor = make_preprocessor()

X_transformed, test_transformed = preprocessor.fit_transform(X), preprocessor.transform(test_df)
model = SVC(probability=True, random_state=42, C=0.6)  
model.fit(X_transformed, y)
preds = model.predict_proba(test_transformed)[:,1]

In [ ]:
#Submission now

output_df = pd.DataFrame({
    'subtaskID':[1,2,2] + [3] * len(test_df),
    'datapointID':[1,1,2] + list(test_df.index),
    'answer':[answer_sub1] + answer_sub2 + list(preds)
})

output_df.head()

,subtaskID,datapointID,answer
0,1,1,1235.000000
1,2,1,0.000000
2,2,2,51.020408
3,3,0,0.228730
4,3,1,0.838892


In [ ]:
output_df.to_csv('submission.csv', index=False)

This aproach works surprisingly well (90/100)

The intended solution is to do pseudo-labeling.